Plot principal functional connectivity gradient (Daniel S. Margulies et al., 2016) on Schaefer 100 left hemishere 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import pearsonr, zscore
import scienceplots
from netneurotools import datasets
from nilearn.datasets import fetch_atlas_schaefer_2018
from nibabel.freesurfer.io import read_annot
plt.style.use(['science', 'nature'])
import seaborn as sns
from matplotlib.colors import ListedColormap
plot_params = {
    "figure.dpi": "600",
    "axes.labelsize": 27,
    "axes.linewidth": 1.5,
    "axes.titlesize": 30,
    "xtick.labelsize": 25,
    "ytick.labelsize": 25,
    "legend.title_fontsize": 15,
    "legend.fontsize": 20,
    "xtick.major.size": 3.5,
    "xtick.major.width": 1.5,
    "xtick.minor.size": 2.5,
    "xtick.minor.width": 1.5,
    "ytick.major.size": 3.5,
    "ytick.major.width": 1.5,
    "ytick.minor.size": 2.5,
    "ytick.minor.width": 1.5,
}
plt.rcParams.update(plot_params)

hier_cmap = LinearSegmentedColormap.from_list(
    "hier_seq",
    ["#f7f7f7", "#cccccc", "#525252"],
    N=256,
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nibabel.freesurfer.io import read_annot
from nilearn import datasets, plotting

fc_grad_vals = pd.read_csv('../../data/processed/schaefer100_LH_fc_grad.csv')['FC_grad']

annot_path = ('../../data/raw/brain_annotations/Schaefer/fsaverage/'
               'atl-Schaefer2018_space-fsaverage_hemi-L_'
               'desc-100Parcels7Networks_deterministic.annot')
labels, ctab, region_names = read_annot(annot_path)

mapped_hierarchy = np.full(labels.shape, np.nan)
for i in range(len(region_names)):
    if i == 0:
        mapped_hierarchy[labels == i] = np.nan
    else:
        mapped_hierarchy[labels == i] = fc_grad_vals[i - 1]

vmin, vmax = np.nanmin(mapped_hierarchy), np.nanmax(mapped_hierarchy)

fs = datasets.fetch_surf_fsaverage('fsaverage')
for view in ['lateral','medial']:
    fig = plt.figure(constrained_layout=True, figsize=(6,6))
    ax = fig.add_subplot(1,1,1, projection='3d')
    plotting.plot_surf_stat_map(
        fs.infl_left, mapped_hierarchy, fs.sulc_left,
        hemi='left', view=view,
        cmap=hier_cmap, colorbar=True,
        symmetric_cbar=False,
        vmin=vmin, vmax=vmax,
        axes=ax, figure=fig,
        engine='matplotlib', avg_method='median'
    )
    for ext in ['pdf']:
        dpi = 300 if ext in ['png'] else None
        fig.savefig(f'plots/FC_grad/fc_grad_{view}.{ext}', dpi=dpi, transparent=True)
    plt.close()

[get_dataset_dir] Dataset found in /home/rad216/nilearn_data/fsaverage
